In this file I will try to see what correlations there are in the training sample

In [41]:
from get_data import *
from matplotlib import pyplot as plt

data = Data(0)

cells = data.get_cell()
particles = data._get_particles()
hits = data.get_hits()
truth = data.get_truth()

cells.head()




,hit_id,ch0,ch1,value
0,1,209,617,0.013832
1,1,210,617,0.079887
2,1,209,618,0.211723
3,2,68,446,0.334087
4,3,58,954,0.034005


In [42]:
hits.head()

,hit_id,x,y,z,volume_id,layer_id,module_id
0,1,-64.4099,-7.163700,-1502.5,7,2,1
1,2,-55.3361,0.635342,-1502.5,7,2,1
2,3,-83.8305,-1.143010,-1502.5,7,2,1
3,4,-96.1091,-8.241030,-1502.5,7,2,1
4,5,-62.6736,-9.371200,-1502.5,7,2,1


In [43]:
particles.head()

,particle_id,vx,vy,vz,px,py,pz,q,nhits
0,4503668346847232,-0.009288,0.009861,-0.077879,-0.055269,0.323272,-0.203492,-1,8
1,4503737066323968,-0.009288,0.009861,-0.077879,-0.948125,0.470892,2.010060,1,11
2,4503805785800704,-0.009288,0.009861,-0.077879,-0.886484,0.105749,0.683881,-1,0
3,4503874505277440,-0.009288,0.009861,-0.077879,0.257539,-0.676718,0.991616,1,12
4,4503943224754176,-0.009288,0.009861,-0.077879,16.439400,-15.548900,-39.824900,1,3


In [44]:
truth.head()

,hit_id,particle_id,tx,ty,tz,tpx,tpy,tpz,weight
0,1,0,-64.4116,-7.164120,-1502.5,250710.000000,-149908.000000,-956385.00000,0.000000
1,2,22525763437723648,-55.3385,0.630805,-1502.5,-0.570605,0.028390,-15.49220,0.000010
2,3,0,-83.8280,-1.145580,-1502.5,626295.000000,-169767.000000,-760877.00000,0.000000
3,4,297237712845406208,-96.1229,-8.230360,-1502.5,-0.225235,-0.050968,-3.70232,0.000008
4,5,418835796137607168,-62.6594,-9.375040,-1502.5,-0.281806,-0.023487,-6.57318,0.000009


In [45]:
# Merge hits with truth to get the particle_id for every hit
hits_with_truth = hits.merge(truth[['hit_id', 'particle_id', 'weight']], on='hit_id')

# Separate signal and noise
signal_hits = hits_with_truth[hits_with_truth['particle_id'] != 0]
noise_hits = hits_with_truth[hits_with_truth['particle_id'] == 0]

print(f"Number of signal hits: {len(signal_hits)}")
print(f"Number of noise hits: {len(noise_hits)}")

Number of signal hits: 103305
Number of noise hits: 17634


In [46]:
# Now lets find what proportions we're dealing with

total_hits = len(hits_with_truth) 

print(f"Proportion of signal hits: {len(signal_hits) / total_hits:.2%}")
print(f"Proportion of noise hits: {len(noise_hits) / total_hits:.2%}")


Proportion of signal hits: 85.42%
Proportion of noise hits: 14.58%


Now lets make sense of the data.

in cells, you have hit_id, ch0, ch1, and value
each hit is picked up by multiple smaller sensors so they have different values and different ch0/ch1

value is the signal amplitude
ch0, ch1 are the coordinates 


Hits:
x, y, z are the coordinates in cartesian form
volume_id - which major group inside the detector it was recorded
layer_id which layer its detected in
module_id which silicon sensor the hit was on



particles and truth used for training

Particles:
particle_id = the unique identifier duh
vx, vy, vz = where the particles were born
px, py, pv = momentum components
q = charge
nhits = number of hits left behind by the particle


Truth
maps particleid to hitid, and gives the true xyz and p(xyz) and values found from simulations without the inaccuracies of the equipments.




In [47]:
# Creating the training data to drop into the code.

training = data.get_cell().merge(data.get_hits(), on="hit_id")
training.head()



,hit_id,ch0,ch1,value,x,y,z,volume_id,layer_id,module_id
0,1,209,617,0.013832,-64.4099,-7.163700,-1502.5,7,2,1
1,1,210,617,0.079887,-64.4099,-7.163700,-1502.5,7,2,1
2,1,209,618,0.211723,-64.4099,-7.163700,-1502.5,7,2,1
3,2,68,446,0.334087,-55.3361,0.635342,-1502.5,7,2,1
4,3,58,954,0.034005,-83.8305,-1.143010,-1502.5,7,2,1


In [48]:
# There are multiple events per hit, fix by grouping by hit_id and aggregating other data


training = training.groupby("hit_id").agg(
    cluster_size=("ch0", "count"),
    charge=("value", "sum"),
    spread_ch0 = ("ch0", lambda x: x.max() - x.min()),
    spread_ch1 = ("ch1", lambda x: x.max() - x.min())
).reset_index() # reset index otherwise we lose the hit_id column

In [49]:
training.head()


,hit_id,cluster_size,charge,spread_ch0,spread_ch1
0,1,3,0.305441,1,1
1,2,1,0.334087,0,0
2,3,8,0.403501,2,5
3,4,1,0.323907,0,0
4,5,1,0.296566,0,0


In [50]:
#oops i got rid of all the other data.. I need to create a new dataframe with better features anyways

training = data.get_hits().merge(training, on="hit_id")
training.head()

,hit_id,x,y,z,volume_id,layer_id,module_id,cluster_size,charge,spread_ch0,spread_ch1
0,1,-64.4099,-7.163700,-1502.5,7,2,1,3,0.305441,1,1
1,2,-55.3361,0.635342,-1502.5,7,2,1,1,0.334087,0,0
2,3,-83.8305,-1.143010,-1502.5,7,2,1,8,0.403501,2,5
3,4,-96.1091,-8.241030,-1502.5,7,2,1,1,0.323907,0,0
4,5,-62.6736,-9.371200,-1502.5,7,2,1,1,0.296566,0,0


In [51]:
import numpy as np

training["r"] = np.sqrt(training["x"] **2 + training["y"] **2)
training["phi"] = np.arctan(training["y"], training["x"])

training.head()


,hit_id,x,y,z,volume_id,layer_id,module_id,cluster_size,charge,spread_ch0,spread_ch1,r,phi
0,1,-64.4099,-7.163700,-1502.5,7,2,1,3,0.305441,1,1,64.807051,-1.432100
1,2,-55.3361,0.635342,-1502.5,7,2,1,1,0.334087,0,0,55.339747,0.566002
2,3,-83.8305,-1.143010,-1502.5,7,2,1,8,0.403501,2,5,83.838292,-0.852033
3,4,-96.1091,-8.241030,-1502.5,7,2,1,1,0.323907,0,0,96.461773,-1.450043
4,5,-62.6736,-9.371200,-1502.5,7,2,1,1,0.296566,0,0,63.370336,-1.464489


In [52]:
is_signal = truth

In [53]:
is_signal.head()


,hit_id,particle_id,tx,ty,tz,tpx,tpy,tpz,weight
0,1,0,-64.4116,-7.164120,-1502.5,250710.000000,-149908.000000,-956385.00000,0.000000
1,2,22525763437723648,-55.3385,0.630805,-1502.5,-0.570605,0.028390,-15.49220,0.000010
2,3,0,-83.8280,-1.145580,-1502.5,626295.000000,-169767.000000,-760877.00000,0.000000
3,4,297237712845406208,-96.1229,-8.230360,-1502.5,-0.225235,-0.050968,-3.70232,0.000008
4,5,418835796137607168,-62.6594,-9.375040,-1502.5,-0.281806,-0.023487,-6.57318,0.000009


In [54]:
is_signal = is_signal.drop(["tx", "ty", "tz", "tpx", "tpy", "tpz", "weight"], axis=1)
is_signal.head()


,hit_id,particle_id
0,1,0
1,2,22525763437723648
2,3,0
3,4,297237712845406208
4,5,418835796137607168


In [55]:
is_signal["particle_id"] = (is_signal["particle_id"] != 0).astype(int)

is_signal.head()


,hit_id,particle_id
0,1,0
1,2,1
2,3,0
3,4,1
4,5,1


That all looks good :)